> Adapted from *Dive into Deep Learning*, section [Adadelta](https://d2l.ai/chapter_optimization/adadelta.html). Not affiliated with or endorsed by the book's authors.
> Source, authors, licenses and changes: [Attribution and changes](#Attribution-and-changes), at the end of this notebook.

Previous: [Section 12.8: RMSProp](12.08_rmsprop.ipynb) | Next: [Section 12.10: Adam](12.10_adam.ipynb)

# Adadelta

Adadelta is yet another variant of AdaGrad ([Section 12.7](https://d2l.ai/chapter_optimization/adagrad.html#sec-adagrad)). The main difference lies in the fact that it decreases the amount by which the learning rate is adaptive to coordinates. Moreover, traditionally it referred to as not having a learning rate since it uses the amount of change itself as calibration for future change. The algorithm was proposed in Zeiler (2012). It is fairly straightforward, given the discussion of previous algorithms so far.

## The Algorithm

In a nutshell, Adadelta uses two state variables, $\mathbf{s}_t$ to store a leaky average of the second moment of the gradient and $\Delta\mathbf{x}_t$ to store a leaky average of the second moment of the change of parameters in the model itself. Note that we use the original notation and naming of the authors for compatibility with other publications and implementations (there is no other real reason why one should use different Greek variables to indicate a parameter serving the same purpose in momentum, Adagrad, RMSProp, and Adadelta).

Here are the technical details of Adadelta. Given the parameter du jour is $\rho$, we obtain the following leaky updates similarly to [Section 12.8](https://d2l.ai/chapter_optimization/rmsprop.html#sec-rmsprop):

$$\begin{aligned}
    \mathbf{s}_t & = \rho \mathbf{s}_{t-1} + (1 - \rho) \mathbf{g}_t^2.
\end{aligned} \tag{12.9.1}$$

The difference to [Section 12.8](https://d2l.ai/chapter_optimization/rmsprop.html#sec-rmsprop) is that we perform updates with the rescaled gradient $\mathbf{g}_t'$, i.e.,

$$\begin{aligned}
    \mathbf{x}_t  & = \mathbf{x}_{t-1} - \mathbf{g}_t'. \\
\end{aligned} \tag{12.9.2}$$

So what is the rescaled gradient $\mathbf{g}_t'$? We can calculate it as follows:

$$\begin{aligned}
    \mathbf{g}_t' & = \frac{\sqrt{\Delta\mathbf{x}_{t-1} + \epsilon}}{\sqrt{{\mathbf{s}_t + \epsilon}}} \odot \mathbf{g}_t, \\
\end{aligned} \tag{12.9.3}$$

where $\Delta \mathbf{x}_{t-1}$ is the leaky average of the squared rescaled gradients $\mathbf{g}_t'$. We initialize $\Delta \mathbf{x}_{0}$ to be $0$ and update it at each step with $\mathbf{g}_t'$, i.e.,

$$\begin{aligned}
    \Delta \mathbf{x}_t & = \rho \Delta\mathbf{x}_{t-1} + (1 - \rho) {\mathbf{g}_t'}^2,
\end{aligned} \tag{12.9.4}$$

and $\epsilon$ (a small value such as $10^{-5}$) is added to maintain numerical stability.



## Implementation

Adadelta needs to maintain two state variables for each variable, $\mathbf{s}_t$ and $\Delta\mathbf{x}_t$. This yields the following implementation.

In [ ]:
import numpy as np

import divelet_ml as d2l
from divelet_ml import optim

def init_adadelta_states(feature_dim):
    s_w, s_b = np.zeros((feature_dim, 1), np.float32), np.zeros(1, np.float32)
    delta_w, delta_b = np.zeros((feature_dim, 1), np.float32), np.zeros(1, np.float32)
    return ((s_w, delta_w), (s_b, delta_b))

def adadelta(params, grads, states, hyperparams):
    rho, eps = hyperparams['rho'], 1e-5
    for p, grad, (s, delta) in zip(params, grads, states):
        # In-place updates via [:]
        s[:] = rho * s + (1 - rho) * grad ** 2
        g = (np.sqrt(delta + eps) / np.sqrt(s + eps)) * grad
        p -= g
        delta[:] = rho * delta + (1 - rho) * g * g

Choosing $\rho = 0.9$ amounts to a half-life time of 10 for each parameter update. This tends to work quite well. We get the following behavior.

In [ ]:
data_iter, feature_dim = d2l.get_data_ch11(batch_size=10)
d2l.train_ch11(adadelta, init_adadelta_states(feature_dim),
               {'rho': 0.9}, data_iter, feature_dim);

For a concise implementation we simply use the Adadelta algorithm from high-level APIs. This yields the following one-liner for a much more compact invocation.

In [ ]:
trainer = optim.Adadelta
d2l.train_concise_ch11(trainer, {'rho': 0.9}, data_iter)

## Summary

* Adadelta has no learning rate parameter. Instead, it uses the rate of change in the parameters itself to adapt the learning rate.
* Adadelta requires two state variables to store the second moments of gradient and the change in parameters.
* Adadelta uses leaky averages to keep a running estimate of the appropriate statistics.

## Exercises

1. Adjust the value of $\rho$. What happens?
1. Show how to implement the algorithm without the use of $\mathbf{g}_t'$. Why might this be a good idea?
1. Is Adadelta really learning rate free? Could you find optimization problems that break Adadelta?
1. Compare Adadelta to Adagrad and RMS prop to discuss their convergence behavior.

Discussions of the original section: [book's forum](https://discuss.d2l.ai/t/1076)

Previous: [Section 12.8: RMSProp](12.08_rmsprop.ipynb) | Next: [Section 12.10: Adam](12.10_adam.ipynb)

## Attribution and changes

Adapted from *Dive into Deep Learning*, section
[Adadelta](https://d2l.ai/chapter_optimization/adadelta.html)
(source: [`chapter_optimization/adadelta.md`](https://github.com/d2l-ai/d2l-en/blob/23d7a5aecceee57d1292c56e90cce307f183bb0a/chapter_optimization/adadelta.md)
at commit `23d7a5aeccee`), by Aston Zhang, Zachary C. Lipton, Mu Li, Alexander J. Smola and contributors.
Text: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Code: adapted from the book's sample code, MIT-0.
Not affiliated with or endorsed by the book's authors.

**Changes from the original** (PyTorch edition):
- Code uses NumPy instead of PyTorch, so it runs in JupyterLite (Pyodide and xeus-python).
- `d2l` is divelet-ml's package (`import divelet_ml as d2l`) instead of the book's `d2l.torch`. `%matplotlib inline` is removed: both JupyterLite kernels display plots inline.
- `adadelta` take the gradients as an argument and update the parameters and states in place, in place of `p.grad`, `torch.no_grad`, `torch.square`, `torch.sqrt` and `p.grad.data.zero_()`; `train_ch11` computes the gradients with autograd. `torch.optim.Adadelta` is `divelet_ml.optim.Adadelta`. The states are `float32` NumPy arrays, in place of `d2l.zeros`.
- The data file is read from this site in JupyterLite, in place of the book's `d2l.download` of its host's copy, and credited here: airfoil self-noise data set, NASA, from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/291/airfoil+self+noise) (Thomas Brooks, D. Pope and Michael Marcolini), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
- Equations keep the book's numbers. Cross-references to other sections link to the book's site, and citations are written inline (author and year); the book's bibliography lists them.
- The d2lbook framework tabs for MXNet, TensorFlow and JAX are removed.